## 7.3 常见陷阱与避免方法

| 陷阱 | 表现 | 避免方法 |
|------|------|----------|
| 数据泄露 | 验证分数虚高，线上效果差 | 不用 ID 作为特征，统计特征只在训练集计算 |
| 过拟合 | 训练分数高，验证分数低 | 使用交叉验证、正则化、early stopping |
| 类别不平衡 | 模型偏向多数类 | 使用 F1 指标、阈值优化、样本均衡 |
| 特征冗余 | 多个高度相关的特征 | 特征选择、相关性分析 |
| 忽略缺失值 | 模型报错或结果异常 | 检查缺失值，合理填充 |

## 7.4 进阶方向

如果你想继续深入，可以探索：

1. **更多特征工程**：
   - 时间序列特征（滑动窗口统计）
   - 文本特征（如果 udmap 有内容）
   - 图特征（用户关系网络）

2. **模型集成**：
   - Bagging：多个模型取平均
   - Stacking：用一个模型学习多个模型的预测结果
   - Blending：类似 Stacking 但更简单

3. **高级调参方法**：
   - 贝叶斯优化（比网格搜索更高效）
   - Optuna 框架

4. **深度学习方法**：
   - 序列模型（LSTM/Transformer）处理用户行为序列
   - 图神经网络处理用户关系

> **最后的话**
>
> 机器学习是一个"做中学"的过程。看 100 篇教程不如自己动手做一个项目。
> 希望这个教程能帮你迈出第一步。祝你学习愉快！

---

# Part 7: 总结与反思

## 7.1 分数演进回顾

| 阶段 | 方法 | F1 分数 | 关键变化 |
|------|------|---------|----------|
| Baseline (官方) | 计数编码 + 目标编码 | ~0.905 | 有数据泄露 |
| RFM 建模 | 用户行为聚合特征 | **0.935** | 修复泄露，真实分数 |
| 网格搜索调优 | 两阶段超参数搜索 | **0.940** | 参数优化 |
| 深度特征工程 | 58 个特征 + 正则化 | **0.962** | 特征 + 正则化 |

## 7.2 关键经验教训

### 经验 1：特征工程 > 模型选择

> 每一次显著的分数提升都来自于更好的特征，而不是更复杂的模型。
> - 从 Baseline 到 RFM：+0.03（特征工程）
> - 从 RFM 到网格搜索：+0.005（参数调优）
> - 从 22 特征到 58 特征：+0.02（深度特征工程）

### 经验 2：数据泄露是最隐蔽的错误

> 如果你的验证分数异常高（比如 0.99），先检查是否有数据泄露。
> 常见泄露源：ID 特征、目标编码、时间穿越。

### 经验 3：理解业务比调参更重要

> 知道"93% 的测试用户在训练集中出现"这个事实，比调 100 组参数更有价值。
> 它直接指导了我们的特征工程策略。

### 经验 4：简单的方法往往最有效

> RFM 模型是一个几十年前的营销模型，但在新用户预测上依然非常有效。
> 不要迷信复杂的算法，先把基础做好。

### 经验 5：阈值优化常被忽视

> 默认阈值 0.5 几乎从来不是最优的。对于不平衡数据，搜索最优阈值是必要的步骤。

In [ ]:
# Enhanced model parameters with regularization
enhanced_params = {
    'objective': 'binary',
    'metric': 'binary_logloss',
    'verbose': -1,
    'n_jobs': 8,
    'seed': 42,
    'max_depth': 15,
    'num_leaves': 255,
    'learning_rate': 0.08,        # Lower learning rate for stability
    'min_child_samples': 20,      # Higher to prevent overfitting
    'feature_fraction': 0.85,
    'bagging_fraction': 0.8,
    'bagging_freq': 3,
    'lambda_l1': 0.1,             # L1 regularization
    'lambda_l2': 0.1,             # L2 regularization
    'min_gain_to_split': 0.02     # Minimum gain required for a split
}

# 5-Fold training with enhanced features
print("=" * 60)
print("Enhanced Model Training (58 features + regularization)")
print("=" * 60)

skf_enhanced = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
oof_enhanced = np.zeros(len(train_df))
test_enhanced = np.zeros(len(test_df))
fold_f1_enhanced = []
thresholds_enhanced = []

for fold, (train_idx, val_idx) in enumerate(skf_enhanced.split(train_df[enhanced_features], train_df['is_new_did'])):
    print(f"Fold {fold+1}/5", end=" ")

    X_train = train_df.iloc[train_idx][enhanced_features]
    y_train = train_df.iloc[train_idx]['is_new_did']
    X_val = train_df.iloc[val_idx][enhanced_features]
    y_val = train_df.iloc[val_idx]['is_new_did']

    dtrain = lgb.Dataset(X_train, label=y_train)
    dval = lgb.Dataset(X_val, label=y_val, reference=dtrain)

    model = lgb.train(
        enhanced_params,
        dtrain,
        num_boost_round=1500,
        valid_sets=[dval],
        callbacks=[lgb.early_stopping(100), lgb.log_evaluation(0)]
    )

    val_pred_proba = model.predict(X_val)
    test_pred_proba = model.predict(test_df[enhanced_features])

    best_th, best_f1 = find_optimal_threshold(y_val, val_pred_proba)
    thresholds_enhanced.append(best_th)
    fold_f1_enhanced.append(best_f1)

    oof_enhanced[val_idx] = val_pred_proba
    test_enhanced += test_pred_proba / 5

    print(f"-> F1: {best_f1:.5f} (threshold={best_th:.2f})")

print(f"\n=== Enhanced Model Results ===")
print(f"Average optimal threshold: {np.mean(thresholds_enhanced):.4f}")
print(f"Fold F1 scores: {[f'{s:.5f}' for s in fold_f1_enhanced]}")
print(f"Average F1: {np.mean(fold_f1_enhanced):.5f}")

# OOF (Out-of-Fold) F1
oof_pred_labels = (oof_enhanced >= np.mean(thresholds_enhanced)).astype(int)
oof_f1 = f1_score(train_df['is_new_did'], oof_pred_labels)
print(f"OOF F1: {oof_f1:.5f}")

## 6.6 带正则化的增强模型

> **正则化是什么？**
>
> 当特征数量增加时，模型更容易过拟合——记住训练数据的噪声而不是学习规律。
>
> 正则化通过以下方式防止过拟合：
> - **L1/L2 正则化**：惩罚过大的模型权重
> - **min_gain_to_split**：要求每次分裂必须带来足够的增益
> - **更低的学习率**：让模型学习得更慢、更稳
>
> 这些调整通常会降低训练集分数，但提高测试集分数——这才是我们真正关心的。

In [ ]:
# Define enhanced feature list (58 features)
enhanced_features = [
    # Original categorical (label encoded)
    'mid', 'eid', 'device_brand', 'ntt', 'operator',
    'common_country', 'common_province', 'common_city',
    'appver', 'channel', 'os_type',
    # Basic time features
    'day', 'dayofweek', 'hour', 'minute', 'ts',
    # Enhanced time features
    'is_weekend', 'is_workday', 'time_period', 'hour_group', 'minute_group',
    # RFM features
    'recency_days', 'frequency_total', 'monetary_unique',
    'first_action_ts', 'last_action_ts',
    'avg_hour', 'hour_consistency', 'earliest_hour', 'latest_hour',
    'avg_dayofweek', 'dayofweek_consistency',
    'action_timespan_seconds', 'action_timespan_days',
    'actions_per_day', 'hour_range',
    # Activity/Recency levels
    'user_activity_level', 'user_recency_level',
    # Diversity features
    'user_device_diversity', 'user_province_diversity',
    'user_city_diversity', 'user_channel_diversity',
    'user_operator_diversity', 'user_os_diversity',
    'user_appver_diversity', 'weekend_activity_ratio',
    # Interaction features
    'device_province_count', 'channel_device_count', 'operator_city_count'
]

# Verify all features exist
missing = [f for f in enhanced_features if f not in train_df.columns]
if missing:
    print(f"Missing features: {missing}")
else:
    print(f"All {len(enhanced_features)} features available!")

In [ ]:
# Label encoding for all categorical features including new ones
cat_features_enhanced = [
    'device_brand', 'ntt', 'operator', 'common_country',
    'common_province', 'common_city', 'appver', 'channel',
    'os_type', 'udmap', 'time_period', 'hour_group', 'minute_group',
    'user_activity_level', 'user_recency_level'
]

label_encoders = {}
for col in cat_features_enhanced:
    le = LabelEncoder()
    combined = pd.concat([train_df[col], test_df[col]], axis=0).astype(str)
    le.fit(combined)
    train_df[col] = le.transform(train_df[col].astype(str))
    test_df[col] = le.transform(test_df[col].astype(str))
    label_encoders[col] = le

print(f"Label encoded {len(cat_features_enhanced)} features")

In [ ]:
# Merge all enhanced features
train_df = train_df.merge(rfm_enhanced, on='did', how='left')
train_df = train_df.merge(user_diversity, on='did', how='left')

# For test: merge with training set user features
test_user_features = pd.concat([
    rfm_enhanced,
    user_diversity.drop('did', axis=1)
], axis=1)

test_df = test_df.merge(rfm_enhanced, on='did', how='left')
test_df = test_df.merge(user_diversity, on='did', how='left')

# Fill NaN with median for numeric, mode for categorical
rfm_numeric_cols = ['recency_days', 'frequency_total', 'monetary_unique',
                    'first_action_ts', 'last_action_ts',
                    'avg_hour', 'hour_consistency', 'earliest_hour', 'latest_hour',
                    'avg_dayofweek', 'dayofweek_consistency',
                    'action_timespan_seconds', 'action_timespan_days',
                    'actions_per_day', 'hour_range']

diversity_cols = ['user_device_diversity', 'user_province_diversity',
                  'user_city_diversity', 'user_channel_diversity',
                  'user_operator_diversity', 'user_os_diversity',
                  'user_appver_diversity', 'weekend_activity_ratio']

for col in rfm_numeric_cols + diversity_cols:
    if col in test_df.columns:
        median_val = train_df[col].median()
        test_df[col] = test_df[col].fillna(median_val)

# Fill categorical features with mode
for col in ['user_activity_level', 'user_recency_level']:
    if col in test_df.columns:
        mode_val = train_df[col].mode()[0]
        test_df[col] = test_df[col].fillna(mode_val)

print(f"Train shape: {train_df.shape}")
print(f"Test shape:  {test_df.shape}")

## 6.5 合并所有特征并训练

> **特征汇总**
>
> 现在我们有三类特征：
> 1. **原始 + 时间特征**：day, dayofweek, hour, is_weekend, time_period 等
> 2. **增强 RFM 特征**（20 个）：recency, frequency, 时间模式, 活跃度等级等
> 3. **多样性 + 交叉特征**：设备多样性, 省份多样性, 设备×省份等
>
> 总计约 58 个特征。让我们合并它们并训练最终模型。

In [ ]:
# Interaction features (computed on training set, applied to both)
# These capture patterns in feature combinations

# Device-location interaction
device_location = train_df.groupby(['device_brand', 'common_province']).size().reset_index(name='device_province_count')

# Channel-device interaction
channel_device = train_df.groupby(['channel', 'device_brand']).size().reset_index(name='channel_device_count')

# Operator-location interaction
operator_location = train_df.groupby(['operator', 'common_city']).size().reset_index(name='operator_city_count')

print("=== Interaction Features ===")
print(f"device_brand x common_province: {len(device_location)} unique combinations")
print(f"channel x device_brand: {len(channel_device)} unique combinations")
print(f"operator x common_city: {len(operator_location)} unique combinations")

# Merge interaction features to train and test
for df in [train_df, test_df]:
    df.merge(device_location, on=['device_brand', 'common_province'], how='left')
    df.merge(channel_device, on=['channel', 'device_brand'], how='left')
    df.merge(operator_location, on=['operator', 'common_city'], how='left')

# Do the merges properly
train_df = train_df.merge(device_location, on=['device_brand', 'common_province'], how='left')
train_df = train_df.merge(channel_device, on=['channel', 'device_brand'], how='left')
train_df = train_df.merge(operator_location, on=['operator', 'common_city'], how='left')

test_df = test_df.merge(device_location, on=['device_brand', 'common_province'], how='left')
test_df = test_df.merge(channel_device, on=['channel', 'device_brand'], how='left')
test_df = test_df.merge(operator_location, on=['operator', 'common_city'], how='left')

# Fill NaN with 1 (unseen combination = count of 1)
for col in ['device_province_count', 'channel_device_count', 'operator_city_count']:
    train_df[col] = train_df[col].fillna(1)
    test_df[col] = test_df[col].fillna(1)

print(f"\nInteraction features added to both train and test sets.")

## 6.4 交叉特征

> **什么是交叉特征？**
>
> 交叉特征是两个或多个特征的组合。例如：
> - `device_brand × common_province`：某个品牌在某个省份的出现次数
> - `channel × device_brand`：某个渠道的某个品牌的用户数量
>
> 这些组合特征可以捕捉到单个特征无法表达的模式。

In [ ]:
# User diversity features (how many different values does each user have?)
user_diversity = train_df.groupby('did').agg({
    'device_brand': 'nunique',     # How many different devices
    'common_province': 'nunique',  # How many different provinces
    'common_city': 'nunique',      # How many different cities
    'channel': 'nunique',          # How many different channels
    'operator': 'nunique',         # How many different operators
    'os_type': 'nunique',          # How many different OS types
    'appver': 'nunique',           # How many different app versions
    'is_weekend': 'mean',          # Weekend activity ratio
}).reset_index()

user_diversity.columns = [
    'did', 'user_device_diversity', 'user_province_diversity',
    'user_city_diversity', 'user_channel_diversity',
    'user_operator_diversity', 'user_os_diversity',
    'user_appver_diversity', 'weekend_activity_ratio'
]

print("=== User Diversity Features ===")
print(f"Features: {list(user_diversity.columns[1:])}")
print(f"\nExample statistics:")
print(user_diversity.describe())

## 6.3 用户多样性特征

> **什么是多样性特征？**
>
> 一个用户可能在多个省份使用过应用，或者用过多个不同品牌的设备。这些"多样性"信息可以区分新老用户：
> - **老用户**：可能换过手机、出过差，接触过多种设备/地区
> - **新用户**：通常只在一台设备、一个地点使用

In [ ]:
# Enhanced RFM features (compute ONLY on training set)
max_ts_train = train_df['ts'].max()
min_ts_train = train_df['ts'].min()

# Core aggregations
rfm_enhanced = train_df.groupby('did').agg({
    'ts': lambda x: (max_ts_train - x.max()).days,  # Recency
    'eid': 'count',                                    # Frequency total
    'mid': 'nunique',                                  # Monetary: unique modules
    'common_ts': ['min', 'max'],                       # First/last action
    'hour': ['mean', 'std', 'min', 'max'],             # Hour patterns
    'dayofweek': ['mean', 'std'],                      # Day patterns
}).reset_index()

# Flatten column names
rfm_enhanced.columns = [
    'did', 'recency_days', 'frequency_total', 'monetary_unique',
    'first_action_ts', 'last_action_ts',
    'avg_hour', 'hour_consistency', 'earliest_hour', 'latest_hour',
    'avg_dayofweek', 'dayofweek_consistency'
]

# Derived features
rfm_enhanced['action_timespan_seconds'] = (
    rfm_enhanced['last_action_ts'] - rfm_enhanced['first_action_ts']
).dt.total_seconds()
rfm_enhanced['action_timespan_days'] = rfm_enhanced['action_timespan_seconds'] / 86400
rfm_enhanced['actions_per_day'] = rfm_enhanced['frequency_total'] / (rfm_enhanced['action_timespan_days'] + 1)
rfm_enhanced['hour_range'] = rfm_enhanced['latest_hour'] - rfm_enhanced['earliest_hour']

# Activity level classification
rfm_enhanced['user_activity_level'] = pd.cut(
    rfm_enhanced['frequency_total'],
    bins=[0, 5, 20, 100, float('inf')],
    labels=['low', 'medium', 'high', 'ultra']
)

# Recency level classification
rfm_enhanced['user_recency_level'] = pd.cut(
    rfm_enhanced['recency_days'],
    bins=[-1, 1, 7, 30, float('inf')],
    labels=['recent', 'this_week', 'this_month', 'long_ago']
)

# Convert datetime to numeric
rfm_enhanced['first_action_ts'] = rfm_enhanced['first_action_ts'].astype(np.int64) // 10**9
rfm_enhanced['last_action_ts'] = rfm_enhanced['last_action_ts'].astype(np.int64) // 10**9

print(f"Enhanced RFM features: {len(rfm_enhanced.columns) - 1} features per user")
print(f"\nFeature list:")
for col in rfm_enhanced.columns:
    if col != 'did':
        print(f"  - {col}")

## 6.2 增强的 RFM 用户特征

> **从 6 个 RFM 特征扩展到 20 个**
>
> 基础 RFM 只有 Recency、Frequency、Monetary 三个维度。我们可以从更多角度刻画用户：
>
> - **时间模式**：用户通常在什么时间段活跃？活跃时间是否规律？
> - **行为密度**：每天平均有多少行为？
> - **行为多样性**：用户使用了多少种不同的功能模块？
> - **活跃度等级**：将用户分为低/中/高/超高活跃度

In [ ]:
# Reload data for enhanced feature engineering
train_df = pd.read_csv("train.csv")
test_df = pd.read_csv("testA_data.csv")

# Convert timestamps
train_df['common_ts'] = pd.to_datetime(train_df['common_ts'], unit='ms')
test_df['common_ts'] = pd.to_datetime(test_df['common_ts'], unit='ms')
train_df['ts'] = train_df['common_ts'].astype(np.int64) // 10**9
test_df['ts'] = test_df['common_ts'].astype(np.int64) // 10**9

for df in [train_df, test_df]:
    # Basic time features
    df['day'] = df['common_ts'].dt.day
    df['dayofweek'] = df['common_ts'].dt.dayofweek
    df['hour'] = df['common_ts'].dt.hour
    df['minute'] = df['common_ts'].dt.minute

    # Weekend/weekday flags
    df['is_weekend'] = (df['dayofweek'] >= 5).astype(int)
    df['is_workday'] = (df['dayofweek'] < 5).astype(int)

    # Time period bins
    df['time_period'] = pd.cut(df['hour'], bins=[0, 6, 12, 18, 24],
                               labels=['dawn', 'morning', 'afternoon', 'evening'])
    df['hour_group'] = pd.cut(df['hour'], bins=[0, 8, 16, 24],
                              labels=['night', 'daytime', 'evening'])
    df['minute_group'] = pd.cut(df['minute'], bins=[0, 15, 30, 45, 60],
                                labels=['0-15', '15-30', '30-45', '45-60'])

print("Enhanced time features created:")
print(f"  is_weekend, is_workday, time_period, hour_group, minute_group")

---

# Part 6: 深度特征工程

> **从 22 个特征到 58 个特征**
>
> 前面我们用 22 个特征达到了 F1 ~0.940。现在我们尝试更深入的特征工程，看看能否进一步提升。
>
> 核心思路：
> 1. **更细粒度的时间特征**：周末/工作日、时间段分组
> 2. **用户多样性特征**：一个用户接触了多少种设备/地区/渠道？
> 3. **交叉特征**：两个特征的组合可能产生新信息
> 4. **用户活跃度分类**：将连续特征离散化为有意义的类别

## 6.1 增强的时间特征

> **为什么需要更细的时间特征？**
>
> 基础的 `hour` 特征是 0-23 的整数，但模型可能不容易学到"凌晨 vs 上午 vs 下午"这样的模式。
>
> 通过创建时间段分组和周末标志，我们可以给模型更明确的提示。

---

# Part 5: 数据泄露 —— 识别与修复

> **什么是数据泄露（Data Leakage）？**
>
> 数据泄露是指在模型训练过程中，**无意中使用了在真实预测时不可能获得的信息**。
>
> 这就像考试时偷看答案——分数很高，但学到的东西是零。
>
> 数据泄露是机器学习中最常见也最隐蔽的错误之一。很多初学者会发现自己的模型在验证集上表现很好，但部署到真实环境后效果大幅下降——十有八九是数据泄露导致的。

## 5.1 本项目中的三种数据泄露

### 泄露类型 1：将用户 ID 作为特征

> 如果把 `did`（用户 ID）直接作为特征，模型会"记住"每个用户：
> - 训练时：`did=12345` → `is_new_did=1`
> - 测试时：看到 `did=12345`，直接输出 1
>
> 这不是学习，是记忆。而且对于训练集中没出现过的新用户，模型完全无法预测。

### 泄露类型 2：在合并数据上计算统计特征

> 如果把训练集和测试集合并（`full_df`），然后计算 RFM 特征：
> - 测试集用户的"最后行为时间"会包含测试集的时间信息
> - 这等于让模型"偷看"了测试集的未来信息
>
> 正确做法：只在训练集上计算，然后通过 `did` 关联到测试集。

### 泄露类型 3：目标编码中的信息泄露

> 当对 `did` 做目标编码时（`did_target = mean(is_new_did)`），实际上是在告诉模型：
> - "这个用户在训练集中的标签是 1 还是 0"
>
> 对于 93% 在测试集中也出现的用户，模型直接就知道答案了。

## 5.2 泄露的影响

> **为什么泄露的分数不可信？**
>
> | 场景 | 有泄露的分数 | 无泄露的分数 | 真实场景分数 |
> |------|-------------|-------------|-------------|
> | 验证集（用户在训练集中） | 0.94 | 0.935 | ~0.935 |
> | 真实环境（全新用户） | 可能很低 | 0.935 | 0.935 |
>
> 泄露模型在"认识的用户"上分数虚高，但在"不认识的用户"上可能完全失效。

## 5.3 如何避免数据泄露

> **三条铁律**：
> 1. **不要把 ID 类字段作为特征**——ID 只用于聚合，不直接入模
> 2. **统计特征只在训练集上计算**——测试集的统计值通过 join 获取
> 3. **目标编码要小心**——对高基数特征（如用户 ID）做目标编码几乎必然泄露
>
> 在本教程的 Part 3 中，我们已经展示了正确的做法。现在你理解了为什么那是正确的。

In [ ]:
# Feature importance analysis
importance_df = pd.DataFrame({
    'feature': features,
    'importance': feature_importance
}).sort_values('importance', ascending=False)

print("=== Feature Importance (Top 15) ===")
print(importance_df.head(15).to_string(index=False))

print(f"\nInsight: action_timespan_seconds is by far the most important feature!")
print(f"New users have short timespans, old users have long timespans.")
print(f"This makes intuitive sense - it's the key signal for distinguishing new vs old users.")

## 4.3 特征重要性分析

> **为什么要看特征重要性？**
>
> 特征重要性告诉我们哪些特征对模型的贡献最大。这有助于：
> 1. **理解模型**：模型在用什么信息做判断？
> 2. **特征选择**：去掉不重要的特征，简化模型
> 3. **指导特征工程**：围绕重要特征做更深入的挖掘

In [ ]:
# Final training with optimized parameters (5-fold)
print("=" * 60)
print("Final Model Training with Optimized Parameters")
print("=" * 60)

skf_final = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
oof_preds_final = np.zeros(len(train_df))
test_preds_final = np.zeros(len(test_df))
fold_f1_final = []
optimal_thresholds_final = []
feature_importance = np.zeros(len(features))

for fold, (train_idx, val_idx) in enumerate(skf_final.split(train_df[features], train_df['is_new_did'])):
    print(f"Fold {fold+1}/5", end=" ")

    X_train = train_df.iloc[train_idx][features]
    y_train = train_df.iloc[train_idx]['is_new_did']
    X_val = train_df.iloc[val_idx][features]
    y_val = train_df.iloc[val_idx]['is_new_did']

    dtrain = lgb.Dataset(X_train, label=y_train)
    dval = lgb.Dataset(X_val, label=y_val, reference=dtrain)

    model = lgb.train(
        final_params,
        dtrain,
        num_boost_round=1000,
        valid_sets=[dval],
        callbacks=[lgb.early_stopping(50), lgb.log_evaluation(0)]
    )

    val_pred_proba = model.predict(X_val)
    test_pred_proba = model.predict(test_df[features])

    best_th, best_f1 = find_optimal_threshold(y_val, val_pred_proba)
    optimal_thresholds_final.append(best_th)
    fold_f1_final.append(best_f1)
    feature_importance += model.feature_importance() / 5

    oof_preds_final[val_idx] = val_pred_proba
    test_preds_final += test_pred_proba / 5

    print(f"-> F1: {best_f1:.5f} (threshold={best_th:.2f})")

print(f"\n=== Final Results ===")
print(f"Average optimal threshold: {np.mean(optimal_thresholds_final):.4f}")
print(f"Fold F1 scores: {[f'{s:.5f}' for s in fold_f1_final]}")
print(f"Average F1: {np.mean(fold_f1_final):.5f}")

## 4.2 用优化后的参数训练最终模型

> **网格搜索的典型结果**
>
> 经过两阶段搜索，通常会发现：
> - **结构参数**：较大的树深度（15）和叶子数（255）效果更好——说明数据有复杂的模式需要捕捉
> - **正则化参数**：适度的采样（feature_fraction=0.9, bagging_fraction=0.8）能防止过拟合
>
> 这些参数组合将 F1 从 0.935 提升到约 **0.940**。

In [ ]:
# Stage 2: Search for regularization parameters
print("=" * 60)
print("Stage 2: Regularization Parameters Search")
print("=" * 60)

stage2_grid = {
    'feature_fraction': [0.6, 0.7, 0.8, 0.9],
    'bagging_fraction': [0.7, 0.8, 0.9],
    'bagging_freq': [3, 5, 7]
}

best_f1_stage2 = 0
best_params_stage2 = {}

param_combinations_2 = list(product(
    stage2_grid['feature_fraction'],
    stage2_grid['bagging_fraction'],
    stage2_grid['bagging_freq']
))

print(f"Total combinations to try: {len(param_combinations_2)}")

for i, (ff, bf, bfreq) in enumerate(param_combinations_2):
    test_params = {
        'objective': 'binary', 'metric': 'binary_logloss', 'verbose': -1,
        'seed': 42,
        **best_params_stage1,  # Use best structure params from stage 1
        'feature_fraction': ff,
        'bagging_fraction': bf,
        'bagging_freq': bfreq
    }

    scores = []
    skf_quick = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
    for train_idx, val_idx in skf_quick.split(train_df[features], train_df['is_new_did']):
        dtrain = lgb.Dataset(train_df.iloc[train_idx][features],
                             label=train_df.iloc[train_idx]['is_new_did'])
        dval = lgb.Dataset(train_df.iloc[val_idx][features],
                           label=train_df.iloc[val_idx]['is_new_did'])
        m = lgb.train(test_params, dtrain, num_boost_round=500,
                      valid_sets=[dval], callbacks=[lgb.early_stopping(30), lgb.log_evaluation(0)])
        pred = (m.predict(train_df.iloc[val_idx][features]) >= 0.5).astype(int)
        scores.append(f1_score(train_df.iloc[val_idx]['is_new_did'], pred))

    avg_f1 = np.mean(scores)
    if avg_f1 > best_f1_stage2:
        best_f1_stage2 = avg_f1
        best_params_stage2 = {'feature_fraction': ff, 'bagging_fraction': bf, 'bagging_freq': bfreq}

print(f"\nStage 2 Best: {best_params_stage2}, F1: {best_f1_stage2:.5f}")

# Combine all best params
final_params = {**best_params_stage1, **best_params_stage2}
final_params.update({'objective': 'binary', 'metric': 'binary_logloss',
                     'verbose': -1, 'seed': 42, 'n_jobs': 8})
print(f"\nFinal optimized parameters: {final_params}")

In [ ]:
# Stage 1: Search for structure parameters
print("=" * 60)
print("Stage 1: Structure Parameters Search")
print("=" * 60)

stage1_grid = {
    'max_depth': [8, 10, 12, 15],
    'num_leaves': [31, 63, 127, 255],
    'learning_rate': [0.05, 0.1],
    'min_child_samples': [10, 20]
}

best_f1_stage1 = 0
best_params_stage1 = {}

# Simple grid search with 3-fold CV (reduced folds for speed)
from itertools import product

param_combinations = list(product(
    stage1_grid['max_depth'],
    stage1_grid['num_leaves'],
    stage1_grid['learning_rate'],
    stage1_grid['min_child_samples']
))

print(f"Total combinations to try: {len(param_combinations)}")

for i, (md, nl, lr, mcs) in enumerate(param_combinations):
    test_params = {
        'objective': 'binary', 'metric': 'binary_logloss', 'verbose': -1,
        'seed': 42, 'max_depth': md, 'num_leaves': nl,
        'learning_rate': lr, 'min_child_samples': mcs
    }

    # Quick 3-fold CV
    scores = []
    skf_quick = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
    for train_idx, val_idx in skf_quick.split(train_df[features], train_df['is_new_did']):
        dtrain = lgb.Dataset(train_df.iloc[train_idx][features],
                             label=train_df.iloc[train_idx]['is_new_did'])
        dval = lgb.Dataset(train_df.iloc[val_idx][features],
                           label=train_df.iloc[val_idx]['is_new_did'])
        m = lgb.train(test_params, dtrain, num_boost_round=500,
                      valid_sets=[dval], callbacks=[lgb.early_stopping(30), lgb.log_evaluation(0)])
        pred = (m.predict(train_df.iloc[val_idx][features]) >= 0.5).astype(int)
        scores.append(f1_score(train_df.iloc[val_idx]['is_new_did'], pred))

    avg_f1 = np.mean(scores)
    if avg_f1 > best_f1_stage1:
        best_f1_stage1 = avg_f1
        best_params_stage1 = {'max_depth': md, 'num_leaves': nl,
                              'learning_rate': lr, 'min_child_samples': mcs}

    if (i + 1) % 16 == 0:
        print(f"  Progress: {i+1}/{len(param_combinations)}, current best F1: {best_f1_stage1:.5f}")

print(f"\nStage 1 Best: {best_params_stage1}, F1: {best_f1_stage1:.5f}")

## 3.7 RFM 模型小结

使用 RFM 特征后，F1 大约在 **0.935** 左右。虽然比 Baseline 的 0.905 看起来低了一些，但这个分数是**真实的**——没有数据泄露。

> **为什么分数反而降低了？**
>
> 因为 Baseline 的高分是靠"作弊"（数据泄露）得到的。修复泄露后分数降低是正常的，说明模型在真正地"学习"而不是"记忆"。

**特征重要性观察**：
- `action_timespan_seconds`（行为时间跨度）是最重要的特征——新用户的时间跨度通常很短
- `first_action_ts`（首次行为时间）也很重要——新用户的首次行为时间更近
- `frequency`（行为频率）——老用户有更多行为记录

---

# Part 4: 模型调优 —— 网格搜索

> **什么是超参数？**
>
> 模型有两类参数：
> - **模型参数**：通过训练数据自动学到的（如决策树的分裂点）
> - **超参数**：需要人工设定的（如树的深度、学习率）
>
> 超参数的选择对模型效果影响很大。网格搜索（Grid Search）是一种暴力搜索方法：尝试所有参数组合，找到最好的那个。

## 4.1 两阶段网格搜索策略

> **为什么要分两阶段？**
>
> 如果一次性搜索所有参数，组合数量会爆炸式增长。分成两阶段：
> 1. **第一阶段**：搜索模型结构参数（树深度、叶子数、学习率等）
> 2. **第二阶段**：固定结构参数，搜索正则化参数（特征采样、样本采样等）
>
> 这样可以大大减少搜索空间，同时找到较好的参数组合。

In [ ]:
# Threshold optimization function
def find_optimal_threshold(y_true, y_pred_proba):
    """Search for the threshold that maximizes F1 score."""
    best_f1 = 0
    best_threshold = 0.5
    for threshold in [0.1, 0.15, 0.2, 0.25, 0.3, 0.35, 0.4, 0.45, 0.5]:
        pred = (y_pred_proba >= threshold).astype(int)
        f1 = f1_score(y_true, pred)
        if f1 > best_f1:
            best_f1 = f1
            best_threshold = threshold
    return best_threshold, best_f1

# LightGBM parameters (manually chosen for now)
params = {
    'objective': 'binary',
    'metric': 'binary_logloss',
    'verbose': -1,
    'n_jobs': 8,
    'seed': 42,
    'max_depth': 15,
    'num_leaves': 255,
    'learning_rate': 0.1,
    'min_child_samples': 10,
    'feature_fraction': 0.9,
    'bagging_fraction': 0.8,
    'bagging_freq': 3
}

# 5-Fold Stratified Cross Validation
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
oof_preds = np.zeros(len(train_df))
test_preds = np.zeros(len(test_df))
fold_f1_scores = []
optimal_thresholds = []

for fold, (train_idx, val_idx) in enumerate(skf.split(train_df[features], train_df['is_new_did'])):
    print(f"=== Fold {fold+1}/5 ===")

    X_train = train_df.iloc[train_idx][features]
    y_train = train_df.iloc[train_idx]['is_new_did']
    X_val = train_df.iloc[val_idx][features]
    y_val = train_df.iloc[val_idx]['is_new_did']

    # Create LightGBM datasets
    dtrain = lgb.Dataset(X_train, label=y_train)
    dval = lgb.Dataset(X_val, label=y_val, reference=dtrain)

    # Train with early stopping
    model = lgb.train(
        params,
        dtrain,
        num_boost_round=1000,
        valid_sets=[dval],
        callbacks=[lgb.early_stopping(50), lgb.log_evaluation(0)]
    )

    # Predict probabilities
    val_pred_proba = model.predict(X_val)
    test_pred_proba = model.predict(test_df[features])

    # Find optimal threshold
    best_th, best_f1 = find_optimal_threshold(y_val, val_pred_proba)
    optimal_thresholds.append(best_th)
    fold_f1_scores.append(best_f1)

    # Store predictions
    oof_preds[val_idx] = val_pred_proba
    test_preds += test_pred_proba / 5

    print(f"  Best threshold: {best_th:.2f}, F1: {best_f1:.5f}")

print(f"\n=== Results ===")
print(f"Average optimal threshold: {np.mean(optimal_thresholds):.4f}")
print(f"Fold F1 scores: {[f'{s:.5f}' for s in fold_f1_scores]}")
print(f"Average F1: {np.mean(fold_f1_scores):.5f}")

## 3.6 阈值优化与 5 折交叉验证

> **为什么要优化阈值？**
>
> 分类模型输出的是概率（0~1），默认以 0.5 为分界线：大于 0.5 预测为正类。
>
> 但对于不平衡数据，0.5 通常不是最优阈值。例如，如果模型对某个样本输出 0.35 的概率，虽然小于 0.5，但这个用户确实可能是新用户。
>
> 通过搜索不同的阈值（0.1, 0.15, 0.2, ...），找到使 F1 最高的那个阈值。

> **5 折交叉验证**
>
> 将训练数据分成 5 份，轮流用 4 份训练、1 份验证。这样每条数据都会被验证一次，得到一个可靠的 F1 估计。

In [ ]:
# Define feature list (22 features, NO did!)
features = [
    # Original categorical features (label encoded)
    'mid', 'eid', 'device_brand', 'ntt', 'operator',
    'common_country', 'common_province', 'common_city',
    'appver', 'channel', 'os_type',
    # Time features
    'day', 'dayofweek', 'hour', 'ts',
    # RFM user features
    'recency', 'frequency', 'mid_nunique',
    'first_action_ts', 'last_action_ts', 'action_timespan_seconds'
]

# Convert datetime columns to numeric for model input
for col in ['first_action_ts', 'last_action_ts']:
    train_df[col] = train_df[col].astype(np.int64) // 10**9
    # For test, these may be NaN - fill with 0
    if col in test_df.columns:
        test_df[col] = pd.to_datetime(test_df[col]).astype(np.int64) // 10**9
    else:
        test_df[col] = 0

print(f"Total features: {len(features)}")
print(f"Features: {features}")

## 3.5 特征列表与模型训练

> **我们准备了哪些特征？**
>
> 共 22 个特征，分为三类：
> 1. **原始特征**（10 个）：device_brand, ntt, operator 等标签编码后的分类特征
> 2. **时间特征**（4 个）：day, dayofweek, hour, ts
> 3. **RFM 用户特征**（6 个）：recency, frequency, mid_nunique, first_action_ts, last_action_ts, action_timespan_seconds
>
> 注意：**`did` 没有被用作特征**——这是避免数据泄露的关键。

In [ ]:
# Label encoding for categorical features
# Fit on combined train+test to ensure consistent encoding
cat_features = ['device_brand', 'ntt', 'operator', 'common_country',
                'common_province', 'common_city', 'appver', 'channel',
                'os_type', 'udmap']

label_encoders = {}
for col in cat_features:
    le = LabelEncoder()
    # Fit on combined unique values from both sets
    combined = pd.concat([train_df[col], test_df[col]], axis=0).astype(str)
    le.fit(combined)
    train_df[col] = le.transform(train_df[col].astype(str))
    test_df[col] = le.transform(test_df[col].astype(str))
    label_encoders[col] = le

print(f"Label encoded {len(cat_features)} features")
print(f"Example: device_brand unique values = {train_df['device_brand'].nunique()}")

## 3.4 标签编码（Label Encoding）

> **什么是标签编码？**
>
> 标签编码将分类特征转换为整数。例如：
> - `"iPhone" → 0`, `"Samsung" → 1`, `"Huawei" → 2`
>
> 与计数编码不同，标签编码只关心类别的"身份"，不关心频率。
>
> **注意**：标签编码假设类别之间没有大小关系。对于有序类别（如"低/中/高"），可能需要特殊处理。

In [ ]:
# Merge RFM features to training set
train_df = train_df.merge(rfm_train, on='did', how='left')

# For test set: merge with training RFM, fill NaN with median
test_rfm = train_df.groupby('did')[['recency', 'frequency', 'mid_nunique',
                                      'action_timespan_seconds']].median().reset_index()

test_df = test_df.merge(test_rfm, on='did', how='left')

# Fill missing values with training set median
rfm_cols = ['recency', 'frequency', 'mid_nunique', 'action_timespan_seconds']
for col in rfm_cols:
    median_val = train_df[col].median()
    test_df[col] = test_df[col].fillna(median_val)

# Also need first_action_ts and last_action_ts for training set
train_df = train_df.merge(
    rfm_train[['did', 'first_action_ts', 'last_action_ts']],
    on='did', how='left'
)

print("=== After Merging ===")
print(f"Train shape: {train_df.shape}")
print(f"Test shape:  {test_df.shape}")
print(f"\nMissing values in RFM features (test):")
print(test_df[rfm_cols].isnull().sum())

## 3.3 特征合并与缺失值处理

> **如何处理测试集中的新用户？**
>
> 测试集中约 7% 的用户在训练集中没有出现过。对于这些用户，我们：
> 1. 通过 `did` 左连接（left join）合并 RFM 特征
> 2. 没有匹配到的用户，用训练集的**中位数**填充
>
> 这是一个合理的默认假设：对于完全陌生的用户，我们假设他的行为模式接近"平均水平"。

In [ ]:
# IMPORTANT: Compute RFM features ONLY on training set
max_ts_train = train_df['ts'].max()

# Aggregate user-level features from training set
rfm_train = train_df.groupby('did').agg({
    'ts': lambda x: (max_ts_train - x.max()).days,  # Recency: days since last action
    'eid': 'count',                                    # Frequency: total actions
    'mid': 'nunique',                                  # Behavior depth: unique modules
    'common_ts': ['min', 'max']                        # First and last action time
})

# Flatten multi-level column names
rfm_train.columns = ['recency', 'frequency', 'mid_nunique', 'first_action_ts', 'last_action_ts']

# Calculate derived features
rfm_train['action_timespan_seconds'] = (
    rfm_train['last_action_ts'] - rfm_train['first_action_ts']
).dt.total_seconds()

# Reset index to make 'did' a column
rfm_train = rfm_train.reset_index()

print("=== RFM Features (Training Set) ===")
print(f"Number of unique users: {len(rfm_train):,}")
print(f"\nFeature statistics:")
print(rfm_train.describe())

## 3.2 RFM 特征构建

> **核心思路**
>
> 我们要为每个用户（`did`）计算一组统计特征，来描述他的行为模式。
>
> 关键点：**只用训练集数据计算**，然后通过 `did` 关联到测试集。
>
> 对于测试集中出现但训练集中没有的新用户（约 7%），我们用训练集的中位数填充——这是一个合理的默认假设。

In [ ]:
# Reload data fresh for proper feature engineering
train_df = pd.read_csv("train.csv")
test_df = pd.read_csv("testA_data.csv")

# Convert timestamps
train_df['common_ts'] = pd.to_datetime(train_df['common_ts'], unit='ms')
test_df['common_ts'] = pd.to_datetime(test_df['common_ts'], unit='ms')

# Create timestamp in seconds for calculations
train_df['ts'] = train_df['common_ts'].astype(np.int64) // 10**9
test_df['ts'] = test_df['common_ts'].astype(np.int64) // 10**9

# Extract time features
for df in [train_df, test_df]:
    df['day'] = df['common_ts'].dt.day
    df['dayofweek'] = df['common_ts'].dt.dayofweek
    df['hour'] = df['common_ts'].dt.hour

print("Data reloaded and time features extracted.")
print(f"Train shape: {train_df.shape}, Test shape: {test_df.shape}")

## 3.1 数据重新加载与准备

为了避免前面 Baseline 中添加的列干扰，我们重新加载数据。这次我们用**正确的方法**来做特征工程。

> **正确做法 vs 错误做法**
>
> | 步骤 | 错误做法 | 正确做法 |
> |------|----------|----------|
> | RFM 计算 | 在 train+test 合并数据上计算 | 只在训练集上计算 |
> | 用户 ID | 直接作为特征 | 不使用（或仅用于聚合） |
> | 测试集处理 | 直接用测试集数据 | 用训练集统计值填充 |

## 2.5 Baseline 小结与反思

Baseline 的 F1 大约在 **0.90** 左右。看起来不错，但我们要问自己一个问题：

> **这个分数真实吗？**

答案是**不真实**。因为我们做了两件"作弊"的事情：

1. **对 `did` 做了目标编码**：相当于直接告诉模型每个用户是新还是老
2. **`did` 本身被作为特征**：模型可以"记住"用户 ID

在真实场景中，我们不可能提前知道测试用户的标签。所以我们需要更诚实的方法——这就是接下来要讲的 RFM 建模。

---

# Part 3: 用户画像与 RFM 建模

> **为什么需要用户画像？**
>
> 前面的 Baseline 虽然分数高，但方法有问题。我们需要一种**可靠的**方式来刻画用户特征。
>
> RFM 模型是市场营销中的经典方法：
> - **R（Recency）**：最近一次行为距今多久？新用户通常最近才活跃
> - **F（Frequency）**：行为频率如何？老用户通常有更多行为记录
> - **M（Monetary）**：行为的"价值"——这里我们用行为的多样性来衡量
>
> **关键原则**：所有用户特征必须**只从训练集计算**，不能看到测试集的信息。

In [ ]:
# Prepare features for baseline model
# Drop non-feature columns: did (ID), udmap (all empty), is_new_did (target), common_ts (already extracted)
drop_cols = ['did', 'udmap', 'is_new_did', 'common_ts']
feature_cols = [c for c in train_df.columns if c not in drop_cols]

print(f"Number of features: {len(feature_cols)}")
print(f"Features: {feature_cols[:10]}... (showing first 10)")

# Cross-validation with HistGradientBoostingClassifier
cv_pred = cross_val_predict(
    HistGradientBoostingClassifier(),
    train_df[feature_cols],
    train_df['is_new_did'],
    cv=5
)

baseline_f1 = f1_score(train_df['is_new_did'], cv_pred)
print(f"\nBaseline F1 Score (with data leakage): {baseline_f1:.4f}")
print("Note: This score is INFLATED due to did_target leakage!")

## 2.4 第一个模型：Baseline

> **模型选择**
>
> 我们先用两个简单的模型来验证特征的效果：
> - **HistGradientBoostingClassifier**：sklearn 自带的梯度提升模型，使用简单
> - **LightGBM**：更专业的梯度提升框架，速度快、效果好
>
> **交叉验证（Cross Validation）**
>
> 为了可靠地评估模型，我们使用交叉验证：把训练数据分成 5 份，每次用 4 份训练、1 份验证，最后取平均分。这样可以避免"运气好"导致的虚高分数。

In [ ]:
# Target encoding: mean of target grouped by each categorical column
# WARNING: This includes did_target which causes data leakage!
for col in cat_cols:
    # Compute mean of target for each category value in training set
    target_mean = train_df.groupby(col)['is_new_did'].mean()
    train_df[col + '_target'] = train_df[col].map(target_mean)
    test_df[col + '_target'] = test_df[col].map(target_mean)

print(f"Created {len(cat_cols)} target-encoded features")

# Show the leakage example
print("\n=== Data Leakage Example: did_target ===")
leakage_example = train_df[['did', 'is_new_did', 'did_target']].head(10)
print(leakage_example)
print("\nNotice: did_target directly reveals the target for known users!")
print("This is why the baseline gets F1 ~0.9 - it's cheating.")

## 2.3 目标编码（Target Encoding）

> **什么是目标编码？**
>
> 目标编码是用该类别对应的**目标变量的平均值**来编码分类特征。
>
> 例如：如果 `device_brand = "BrandX"` 的用户中有 30% 是新用户，那么 `device_brand_target = 0.3`。
>
> **为什么有效？** 它直接捕捉了类别和目标之间的关系。
>
> **但是！这里有一个严重的陷阱——数据泄露：**
>
> 当我们对 `did`（用户 ID）做目标编码时，实际上是告诉模型："这个用户在训练集中是新用户还是老用户"。对于 93% 在测试集中也出现的用户，模型直接就知道答案了！
>
> 这就是为什么下面的 Baseline 能得到 0.9 的高分——它在"作弊"。

In [ ]:
# Count encoding for categorical features
# For each categorical column, replace the value with its frequency count
cat_cols = ['mid', 'eid', 'did', 'device_brand', 'ntt', 'operator',
            'common_country', 'common_province', 'common_city',
            'appver', 'channel', 'os_type']

for col in cat_cols:
    # Count occurrences in training set, then map to both train and test
    counts = train_df[col].value_counts()
    train_df[col + '_count'] = train_df[col].map(counts)
    test_df[col + '_count'] = test_df[col].map(counts)

print(f"Created {len(cat_cols)} count features")
print(f"Example: device_brand_count for first 5 rows:")
print(train_df[['device_brand', 'device_brand_count']].head())

## 2.2 计数编码（Count Encoding）

> **什么是计数编码？**
>
> 对于分类特征（如设备品牌、运营商），模型无法直接理解文字。我们需要把它们转换成数字。
>
> 计数编码是一种简单但有效的方法：用该类别的**出现次数**来代替类别本身。
>
> 例如：`device_brand = "iPhone"` → `device_brand_count = 50000`（表示 iPhone 在数据中出现了 5 万次）
>
> **为什么有用？** 高频类别可能和目标变量有某种关联。比如某个小众品牌的用户可能都是新用户。

In [ ]:
# Time features are already extracted in Part 1.4
# Let's verify they exist and look at their distribution
print("=== Time Features Summary ===")
print(f"Day of month: sorted values = {sorted(train_df['day'].unique())}")
print(f"Day of week:  {sorted(train_df['dayofweek'].unique())} (0=Mon, 6=Sun)")
print(f"Hours:        {sorted(train_df['hour'].unique())}")

# Distribution of new vs old users by hour
print("\n=== New User Ratio by Hour ===")
hour_stats = train_df.groupby('hour')['is_new_did'].agg(['mean', 'count'])
hour_stats.columns = ['new_user_ratio', 'sample_count']
print(hour_stats.head(10))

print("\nInsight: New user ratio varies by hour - this could be a useful feature!")

---

# Part 2: 基础特征工程

> **什么是特征工程？**
>
> 特征工程是将原始数据转换为模型可以理解的数值特征的过程。它是机器学习项目中**最重要**的环节——好的特征比复杂的模型更能提升效果。
>
> 打个比方：如果你要判断一个人是否是新员工，你会看什么？
> - 他在公司待了多久？（时间特征）
> - 他认识多少同事？（社交特征）
> - 他每天几点上班？（行为模式）
>
> 这些"看什么"就是特征工程要做的事。

## 2.1 时间特征提取

从时间戳中提取有意义的特征：月、日、小时、星期几。这些特征可以捕捉用户行为的时间模式。

## 1.6 数据探索小结

通过上面的探索，我们发现了几个重要信息：

| 发现 | 影响 | 应对策略 |
|------|------|----------|
| 类别不平衡（84% 老用户） | 模型可能偏向预测为老用户 | 使用 F1 作为评估指标，而非 accuracy |
| 93% 测试用户在训练集中出现 | 用户级特征会非常有效 | 构建用户行为聚合特征（RFM） |
| 数据时间跨度约 1 个月 | 时间特征有区分度 | 提取时间相关特征 |
| udmap 全部为空 | 该字段无信息量 | 可以丢弃 |

> **初学者要点**：数据探索不是走形式，每一步发现都会影响后续的建模策略。花时间理解数据，远比盲目调参更有价值。

In [ ]:
# Analyze udmap field
train_empty = (train_df['udmap'] == '{}').sum()
test_empty = (test_df['udmap'] == '{}').sum()

print("=== udmap Analysis ===")
print(f"Train: {train_empty:,} / {len(train_df):,} are empty ({train_empty/len(train_df):.2%})")
print(f"Test:  {test_empty:,} / {len(test_df):,} are empty ({test_empty/len(test_df):.2%})")

# Sample some non-empty entries
non_empty = train_df[train_df['udmap'] != '{}']['udmap'].head(5)
print(f"\nNon-empty samples (if any): {len(non_empty)}")
if len(non_empty) > 0:
    for i, val in enumerate(non_empty):
        print(f"  {i}: {val}")

print("\nInsight: udmap is completely empty in this dataset.")
print("We can safely drop it or create a binary 'is_empty' flag.")

## 1.5 udmap 字段分析

> **udmap** 是一个 JSON 格式的字段，理论上可以包含 `botId`（机器人 ID）和 `pluginId`（插件 ID）等信息。
>
> 但实际数据中，这个字段可能是空的。我们需要验证一下，因为：
> - 如果大部分是空的，那这个字段基本没用
> - 如果有数据，解析 JSON 可以提取额外特征

In [ ]:
# Convert timestamp from milliseconds to datetime
train_df['common_ts'] = pd.to_datetime(train_df['common_ts'], unit='ms')
test_df['common_ts'] = pd.to_datetime(test_df['common_ts'], unit='ms')

# Time range analysis
print("=== Time Range Analysis ===")
print(f"Training set: {train_df['common_ts'].min()} ~ {train_df['common_ts'].max()}")
print(f"Test set:     {test_df['common_ts'].min()} ~ {test_df['common_ts'].max()}")

# Extract basic time features for exploration
train_df['ts'] = train_df['common_ts'].astype(np.int64) // 10**9  # to seconds
test_df['ts'] = test_df['common_ts'].astype(np.int64) // 10**9

train_df['day'] = train_df['common_ts'].dt.day
train_df['dayofweek'] = train_df['common_ts'].dt.dayofweek
train_df['hour'] = train_df['common_ts'].dt.hour

test_df['day'] = test_df['common_ts'].dt.day
test_df['dayofweek'] = test_df['common_ts'].dt.dayofweek
test_df['hour'] = test_df['common_ts'].dt.hour

print(f"\nTraining set spans ~{(train_df['common_ts'].max() - train_df['common_ts'].min()).days} days")
print(f"Day of week range: {train_df['dayofweek'].min()} (Mon) to {train_df['dayofweek'].max()} (Sun)")
print(f"Hour range: {train_df['hour'].min()}:00 to {train_df['hour'].max()}:00")

## 1.4 时间特征分析

> **为什么时间特征重要？**
>
> 用户行为具有很强的时间规律：
> - 新用户通常在某个时间点首次出现，行为时间跨度很短
> - 老用户有长期的行为历史，时间跨度长
> - 不同时间段（工作日/周末、白天/晚上）的用户行为模式不同
>
> `common_ts` 字段是毫秒级时间戳，需要转换为可读的日期时间格式，然后提取有用的特征。

In [ ]:
# DID overlap analysis between train and test
train_dids = set(train_df['did'].unique())
test_dids = set(test_df['did'].unique())
overlap_dids = train_dids & test_dids

print("=== DID Overlap Analysis ===")
print(f"Unique DIDs in train: {len(train_dids):,}")
print(f"Unique DIDs in test:  {len(test_dids):,}")
print(f"Overlapping DIDs:     {len(overlap_dids):,}")
print(f"\n{len(overlap_dids)/len(test_dids):.2%} of test DIDs appear in training set")
print(f"{len(overlap_dids)/len(train_dids):.2%} of train DIDs appear in test set")

print(f"\nInsight: 93% of test users are also in training set!")
print(f"This means user-level features will be very powerful,")
print(f"but we must be careful about data leakage.")

## 1.3 用户 ID 重叠分析

> **这是一个非常关键的发现！**
>
> 我们需要检查训练集和测试集中的用户 ID（`did`）有多少是重叠的。这个信息直接影响后续的特征工程策略：
> - 如果大量用户在两个集合中都出现，那么基于用户 ID 的特征（如历史行为统计）会非常有效
> - 但同时也要警惕：如果直接把 `did` 作为特征，模型可能会"记住"用户而不是"学习"规律——这就是**数据泄露**

In [ ]:
# Target variable distribution
target_counts = train_df['is_new_did'].value_counts()
target_ratio = train_df['is_new_did'].value_counts(normalize=True)

print("=== Target Distribution ===")
print(f"Old users (0): {target_counts[0]:,} ({target_ratio[0]:.2%})")
print(f"New users (1): {target_counts[1]:,} ({target_ratio[1]:.2%})")
print(f"\nTotal training samples: {len(train_df):,}")
print(f"\nInsight: ~84% are old users, ~16% are new users")
print(f"Baseline accuracy if always predicting 'old': {target_ratio[0]:.2%}")

## 1.2 目标变量分布分析

> **为什么要看目标变量分布？**
>
> 分类问题中，如果正负样本比例悬殊（比如 99% vs 1%），模型可能会"偷懒"——直接预测所有样本为多数类， accuracy 依然很高但毫无用处。这叫**类别不平衡**问题。
>
> 了解分布后，我们可以：
> - 决定是否需要做样本均衡（过采样/欠采样）
> - 选择合适的评估指标（F1 比 accuracy 更适合不平衡数据）
> - 理解模型的"基准表现"（如果全部预测为老用户，准确率就有 84.4%）

**数据字段说明**：

| 字段 | 含义 |
|------|------|
| `mid` | 用户行为模块 ID |
| `eid` | 用户行为事件 ID |
| `did` | 用户/设备 ID（Device ID） |
| `device_brand` | 设备品牌 |
| `ntt` | 网络类型 |
| `operator` | 运营商 |
| `common_country` | 国家 |
| `common_province` | 省份 |
| `common_city` | 城市 |
| `appver` | 应用版本 |
| `channel` | 渠道 |
| `common_ts` | 时间戳（毫秒） |
| `os_type` | 操作系统类型（Android/iOS） |
| `udmap` | 用户行为映射（JSON 格式） |
| `is_new_did` | **目标变量**：是否为新用户（仅训练集有） |

In [ ]:
# Load training and test data
train_df = pd.read_csv("train.csv")
test_df = pd.read_csv("testA_data.csv")

print(f"Training set shape: {train_df.shape}")
print(f"Test set shape: {test_df.shape}")
print(f"\nTraining set columns:\n{list(train_df.columns)}")

---

# Part 1: 数据加载与探索

> **为什么要先探索数据？**
>
> 很多初学者拿到数据就直接开始建模，这是一个常见错误。数据探索（EDA, Exploratory Data Analysis）能帮你：
> 1. **理解数据结构**：知道有哪些字段、什么类型、有多少数据
> 2. **发现数据问题**：缺失值、异常值、数据不一致
> 3. **找到建模线索**：哪些特征可能有用、数据分布如何
> 4. **避免后续踩坑**：比如类别不平衡、数据泄露等

## 1.1 加载数据

首先加载训练集和测试集，看看数据的基本信息。

In [ ]:
# Core libraries
import pandas as pd
import numpy as np

# Machine learning
import lightgbm as lgb
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import f1_score
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import HistGradientBoostingClassifier

# Utilities
import json
import warnings
warnings.filterwarnings('ignore')

print("All libraries imported successfully!")

---

# Part 0: 环境准备

在开始之前，我们需要导入所有需要的库。下面逐一介绍每个库的用途：

- **pandas**：数据处理的核心库，类似 Excel 但更强大
- **numpy**：数值计算库，处理数组和数学运算
- **lightgbm**：微软开发的梯度提升框架，速度快、效果好，是竞赛利器
- **sklearn**：机器学习工具库，提供模型评估、交叉验证、预处理等功能
- **warnings**：控制警告信息的显示

> **初学者提示**：你不需要记住所有库的用法。先知道它们能做什么，用到时再查文档。

# 新用户预测 —— 机器学习入门实战教程

## 项目背景

本教程基于**科大讯飞 2025 年新用户预测竞赛**，目标是根据用户的历史行为数据，预测一个用户是否为**新用户**（二分类问题）。

**评估指标**：F1 Score（精确率和召回率的调和平均）

## 你将学到什么

本教程将带你走完一个完整的机器学习项目流程：

| 阶段 | 内容 | 对应章节 |
|------|------|----------|
| 数据探索 | 了解数据结构、分布、潜在陷阱 | Part 1 |
| 基础特征工程 | 时间特征、计数编码、目标编码 | Part 2 |
| 用户画像建模 | RFM 模型、用户行为聚合特征 | Part 3 |
| 模型调优 | 网格搜索、超参数优化 | Part 4 |
| 问题修复 | 数据泄露的识别与修复 | Part 5 |
| 深度特征工程 | 交叉特征、多样性特征 | Part 6 |
| 总结反思 | 经验教训与进阶方向 | Part 7 |

## 适合谁

- 有一点 Python/pandas 基础，但刚接触机器学习的同学
- 想了解特征工程实际操作的同学
- 想知道"为什么我的模型分数虚高"的同学

> **免责声明**：本教程记录的是学习过程，很多做法是通过 AI 辅助完成的。重点在于展示思路演进，而非追求最优方案。